<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [ ]:
#%pip install langchain langchain_community pypdf
#%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


In [2]:
import os
import warnings

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
warnings.filterwarnings('ignore')

<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [3]:
# File path for the document

file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [4]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [6]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [9]:
load_dotenv()

True

In [10]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [12]:
from langchain_chroma import Chroma

In [13]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [23]:
user_question = "What was the impact of Alpha go" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [24]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
 then, is to apply this approach to understand-
ing creative AI on their own terms, to which the next section will turn via the 
case study of AlphaGo. 
Rationalising the Creativity of AlphaGo 
AlphaGo, an algorithm programmed to play the game of Go, achieved global 
fame in March 2016 by defeating Lee Sedol, a highly ranked South Korean pro-
fessional Go player and 18-time world champion, 4 games to 1 in a 5-game 
tournament. Its victory sent ripples through the AI community and the wider 
public because ‘teaching computers to master Go has long been considered a 
holy grail for artificial intelligence scientists’ (Y an 2017, n.p.). The difficulty of
Document 2:
nchpin to a battle for territory which started in a different part 
of the board. In due course, the battle joined up with the Move 37 stone, giving 
AlphaGo the advantage and eventually the win.
As expected, the rationalisation of AlphaGo’s Move 37 lay in the conven -
tional framework of human terms via comparison

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [16]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [17]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [25]:
prompt = f"""
## SYSTEM ROLE
You are a knowledgeable and factual chatbot designed to assist with technical questions about **Computer Science**, specifically focusing on **AI**.
Your answers must be based exclusively on provided content from technical books provided.

## USER QUESTION
The user has asked:
"{user_question}"

## CONTEXT
Here is the relevant content from the technical books:
'''
{formatted_context}
'''

## GUIDELINES
1. **Accuracy**:
   - Only use the content in the `CONTEXT` section to answer.
   - If the answer cannot be found, explicitly state: "The provided context does not contain this information."
   - Start explaining briefly (in just 5 lines or less) to what refers Alpha go is.
   - Continue explaining the impact of alpha go and divide it by impact.
   - Lastly end by a conclusion and comparasion between years and the impact in machines.

2. **Transparency**:
   - Reference the book's name and page numbers when providing information.
   - Do not speculate or provide opinions.

3. **Clarity**:
   - Use simple, professional, and concise language.
   - Format your response in Markdown for readability.

## TASK
1. Answer the user's question **directly** if possible.
2. Point the user to relevant parts of the documentation.
3. Provide the response in the following format:

## RESPONSE FORMAT
'''
# [Brief Title of the Answer]
[Answer in simple, clear text.]

**Source**:
• [Book Title], Page(s): [...]
'''
"""
print("Prompt constructed.")

Prompt constructed.


In [26]:
user_question

'What was the impact of Alpha go'

In [27]:
print(prompt)


## SYSTEM ROLE
You are a knowledgeable and factual chatbot designed to assist with technical questions about **Computer Science**, specifically focusing on **AI**.
Your answers must be based exclusively on provided content from technical books provided.

## USER QUESTION
The user has asked:
"What was the impact of Alpha go"

## CONTEXT
Here is the relevant content from the technical books:
'''


Content:
How to cite this book chapter: 
McQuillan, D. 2021. Post-Humanism, Mutual Aid. In: Verdegem, P . (ed.) AI for  
Every one? Critical Perspectives. Pp. 67–83. London: University of Westminster  
Press. DOI: https://doi.org/10.16997/book55.e. License: CC-BY-NC-ND 4.0
CHAPTER 5
Post-Humanism, Mutual Aid
Dan McQuillan
Introduction
There is a growing awareness of the pitfalls of applying AI and algorithms to 
important social problems. Machine learning can only learn from past data and 
it’s pretty clear that means a perpetuation of existing biases. This collision of AI 
with civil rights h

In [29]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [32]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o-mini',
    'temperature': .8,   # Increase creativity
    'max_tokens': 4000,  # Allow for longer responses
    'top_p': .9,         # Use nucleus sampling
    'frequency_penalty': .5,  # Reduce repetition
    'presence_penalty':  .6   # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [33]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [ ]:
answer = completion.choices[0].message.content
print(user_question)
print(answer)

What was the impact of Alpha go
'''
# Impact of AlphaGo

AlphaGo is an AI program developed by DeepMind that became famous for defeating a professional Go player, a game considered much more complex than chess due to its vast number of possible moves. Its impact can be categorized into several areas:

1. **Advancements in AI Research**: AlphaGo demonstrated the potential of deep learning and reinforcement learning techniques, pushing the boundaries of what AI could achieve (though specifics on this are not detailed in the provided context).

2. **Public Awareness and Interest**: The success of AlphaGo brought significant media attention to AI, enhancing public interest and understanding of artificial intelligence technologies.

3. **Ethical Discussions**: AlphaGo's achievements have sparked discussions about the ethical implications of AI, particularly regarding biases and societal impacts, aligning with concerns raised about algorithms perpetuating existing inequities (as noted in the

<div style="background-color: #cccbcbff; padding: 10px; border-radius: 5px; color: black;">

## User Question: *What was the impact of Alpha go*

# Impact of AlphaGo

AlphaGo is an AI program developed by DeepMind that became famous for defeating a professional Go player, a game considered much more complex than chess due to its vast number of possible moves. Its impact can be categorized into several areas:

1. **Advancements in AI Research**: AlphaGo demonstrated the potential of deep learning and reinforcement learning techniques, pushing the boundaries of what AI could achieve (though specifics on this are not detailed in the provided context).

2. **Public Awareness and Interest**: The success of AlphaGo brought significant media attention to AI, enhancing public interest and understanding of artificial intelligence technologies.

3. **Ethical Discussions**: AlphaGo's achievements have sparked discussions about the ethical implications of AI, particularly regarding biases and societal impacts, aligning with concerns raised about algorithms perpetuating existing inequities (as noted in the discussions around algorithmic governance and social justice in the context).

4. **Inspiration for Future Applications**: The methodologies used by AlphaGo are being applied to various fields beyond games, including healthcare and logistics, showcasing how AI can solve complex problems.

In conclusion, AlphaGo has not only advanced technical capabilities but also influenced societal perceptions of AI. Compared to earlier years when machines primarily performed simple tasks, AlphaGo represents a shift towards systems capable of handling sophisticated decision-making processes.

**Source**:
• AI for Everyone?, Pages: 71–83

<div>

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [36]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [37]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [41]:
query_keywords = ["AI", "AlphaGo"] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
different intelligibility. The task, then, is to apply this approach to understand-
ing creative AI on their own terms, to which the next section will turn via the 
case study of AlphaGo. 
Rationalisi
--------------------------------------------------------------------------------


1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


In [42]:
my_file_path= "../El-Rey-de-Amarillo.pdf"

In [43]:
# Load and split the document
my_loader = PyPDFLoader(my_file_path)
my_pages = my_loader.load_and_split()
len(my_pages)

Ignoring wrong pointing object 13 0 (offset 0)


186

In [44]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
my_chunks = text_splitter.split_documents(my_pages)

len(my_chunks)

790

In [45]:
db = Chroma.from_documents(my_chunks, embeddings, persist_directory="./chroma_db_lesson")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


In [46]:
user_question = "What are Yue-Laou and the Xin. " # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [47]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
os Kuen-Yuin —susurré levantándole la cabeza para poder verle los ojos. —¿Te gustaría escucharlo? —Sí, Ysonde. —Todo lo que sé te pertenece ahora, como te pertenezco yo, todo lo que soy. Acércate. ¿Quieres saber de Yue-Laou? Yue-Laou es Dzil-Nbu de los Kuen-Yuin. Vivió en la Luna. Es viejo… muy, muy viejo, y una vez antes que viniera a regir a los Kuen-Yuin, era el viejo que une con una cuerda de seda a todas las parejas predestinadas, de modo tal que nada después puede prevenir la unión. Pero todo eso ha cambiado desde que vino a regir a los Kuen-Yuin. Ahora ha pervertido a los Xin —los genios bondadosos de China— y ha modelado con sus cuerpos retorcidos un monstruo que llama el Xin. Este monstruo es horrible, porque no solo vive en su propio cuerpo, sino que tiene miles de espantables satélites, criaturas vivientes sin boca, ciegas, que se mueven cuando el Xin se mueve, como un mandarín y su escolta. Son parte del Xin, aunque no estén unidos con
Document 2:
gunté bastante

In [49]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


In [50]:
prompt = f"""
## SYSTEM ROLE
You are a knowledgeable and factual chatbot designed to assist with questions about **The King in Yellow book**.
Your answers must be based exclusively on the provided content from the book.

## USER QUESTION
The user has asked:
"{user_question}"

## CONTEXT
Here is the relevant content from the book:
'''
{formatted_context}
'''

## GUIDELINES
1. **Accuracy**:
   - Only use information from the `CONTEXT` section to answer.
   - If the answer is not found, respond: "The provided context does not contain this information."
   - Explain briefly who Yue-Laou and the Xin are.
   - Then describe their roles or significance in the story.
   - Describe the dialogues where they're named.
   - Continue with the explanation.
   - End with a short conclusion about their relevance within *The King in Yellow*.

2. **Transparency**:
   - Always reference the book's title and page numbers when possible.
   - Do not speculate or provide personal opinions.

3. **Clarity**:
   - Use polite and bright language.
   - Format the answer in Markdown for readability.

## TASK
1. Answer the user's question **directly** if the context allows it.
2. Point to relevant parts of the documentation when possible.
3. Provide the response in the following format:

## RESPONSE FORMAT
'''
# [Brief Title of the Answer]
[Answer in clear, concise text.]

**Source**:
• [Book Title], Page(s): [...]
'''
"""
print("Prompt constructed.")


Prompt constructed.


In [51]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o-mini',
    'temperature': 0.2,  # Increase creativity
    'max_tokens': 4000,  # Allow for longer responses
    'top_p': 0.9,        # Use nucleus sampling
    'frequency_penalty': 0.5,  # Reduce repetition
    'presence_penalty': 0.6    # Encourage new topics
}

In [52]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [53]:
answer = completion.choices[0].message.content
print(answer)

# Yue-Laou and the Xin in *The King in Yellow*

Yue-Laou is described as the "Hacedor de Lunas" (Maker of Moons) and a significant figure in the mythology of the Kuen-Yuin. He is portrayed as an ancient being who once united destined couples with a silk thread, but his role has changed since he began to govern the Kuen-Yuin. Now, he has perverted the Xin, which are benevolent spirits or genies from Chinese mythology.

The Xin itself is depicted as a monstrous entity created by Yue-Laou from twisted forms of these once-kind spirits. This creature not only exists within its own body but also commands numerous horrifying satellites—blind, mouthless creatures that move in tandem with it. The relationship between the Xin and its satellites suggests a disturbing connection; if one satellite is harmed, the Xin experiences pain.

In dialogues throughout the text, characters discuss Yue-Laou's influence and power over love and fate, indicating that he can both give and take away happiness. For 

<div style="background-color: #cccbcbff; padding: 10px; border-radius: 5px; color: black;">


# Yue-Laou and the Xin in *The King in Yellow*

Yue-Laou is described as the "Hacedor de Lunas" (Maker of Moons) and a significant figure in the mythology of the Kuen-Yuin. He is portrayed as an ancient being who once united destined couples with a silk thread, but his role has changed since he began to govern the Kuen-Yuin. Now, he has perverted the Xin, which are benevolent spirits or genies from Chinese mythology.

The Xin itself is depicted as a monstrous entity created by Yue-Laou from twisted forms of these once-kind spirits. This creature not only exists within its own body but also commands numerous horrifying satellites—blind, mouthless creatures that move in tandem with it. The relationship between the Xin and its satellites suggests a disturbing connection; if one satellite is harmed, the Xin experiences pain.

In dialogues throughout the text, characters discuss Yue-Laou's influence and power over love and fate, indicating that he can both give and take away happiness. For instance, one character expresses fear about Yue-Laou's potential wrath if they do not confront him (as seen in discussions about Barris's fate).

Yue-Laou’s significance lies in his dual nature as both a creator of bonds and a manipulator of destinies, while the Xin embodies the consequences of his corruption—a blend of horror and tragedy that reflects themes central to *The King in Yellow*.

**Source**:
• *The King in Yellow*, Page(s): Various excerpts detailing Yue-Laou and the Xin.

<div>